# Phase 1 Notebook

这个 notebook 按 `project_plan_v2.md` 的阶段一执行到 **Task 5: CPU Reference**，把阶段一的关键产物和输出集中在一个文件里：

1. 环境验证与模型加载
2. 层结构分析与 `layer_list.csv`
3. FP16 baseline 与 `baseline_fp16.csv` / `baseline_results.csv`
4. INT8 量化与 `quantized_weights.pt`
5. CPU reference correctness 检查

## Task 1: 环境验证 + 模型加载

In [1]:
import csv
import platform
from collections import Counter
from pathlib import Path
from pprint import pprint

import accelerate
import torch
import transformers

from cpu_reference import check_correctness, quantized_matmul_reference, quantized_matvec_reference, self_test
from phase1_utils import (
    MODEL_DIR,
    benchmark_single_layer_fp16,
    build_baseline_rows,
    collect_layer_rows,
    generate_text_smoke,
    load_layer_types,
    load_model_and_tokenizer,
    resolve_module,
    save_layer_rows_csv,
    select_phase1_benchmark_rows,
)
from quantize import (
    attach_layer_metadata,
    load_quantized,
    quantize_layer_records,
    save_quantized,
    summarize_errors_by_type,
)

/home/haozhong/vllm-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print('python:', platform.python_version())
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('accelerate:', accelerate.__version__)

model, tokenizer, config = load_model_and_tokenizer(
    MODEL_DIR,
    torch_dtype=torch.float16,
    device_map='auto',
)
model_device = next(model.parameters()).device

print('model_dir:', MODEL_DIR)
print('config_class:', type(config).__name__)
print('model_class:', type(model).__name__)
print('device:', model_device)
print('architectures:', getattr(config, 'architectures', None))

python: 3.12.3
torch: 2.9.1+cu128
transformers: 5.6.0.dev0
accelerate: 1.13.0


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/723 [00:00<02:46,  4.32it/s]

Loading weights:   6%|▌         | 45/723 [00:00<00:04, 165.12it/s]

...

Loading weights:  54%|█████▍    | 393/723 [00:01<00:00, 443.82it/s]

Loading weights:  67%|██████▋   | 488/723 [00:01<00:00, 588.13it/s]

Loading weights:  99%|█████████▉| 716/723 [00:01<00:00, 1060.61it/s]

Loading weights: 100%|██████████| 723/723 [00:01<00:00, 560.09it/s] 

model_dir: /home/haozhong/ECE9483/models/Qwen3.5-4B
config_class: Qwen3_5Config
model_class: Qwen3_5ForConditionalGeneration
device: cuda:0
architectures: ['Qwen3_5ForConditionalGeneration']


In [3]:
smoke_output = generate_text_smoke(model, tokenizer, prompt='Hello, how are you?', max_new_tokens=10)
print('smoke_output:', smoke_output)

smoke_output: Hello! I'm doing well, thanks for asking


## Task 2: 层结构分析

In [4]:
layer_types = load_layer_types(MODEL_DIR)
rows = collect_layer_rows(model, layer_types)
layer_list_path = save_layer_rows_csv(rows, Path('layer_list.csv'))
benchmark_rows = select_phase1_benchmark_rows(rows)

print('total_target_layers:', len(rows))
print('sub_type_counts:', Counter(row['sub_type'] for row in rows))
print('priority_counts:', Counter(row['priority'] for row in rows))
print('saved_layer_list:', layer_list_path.resolve())
print()
print('representative benchmark rows:')
for row in benchmark_rows:
    print(
        f"- {row['benchmark_name']:22s} | {row['sub_type']:13s} | "
        f"{row['proj_name']:12s} | {row['shape']:14s} | {row['submodule_name']}"
    )

total_target_layers: 248
sub_type_counts: Counter({'DeltaNet_Attn': 120, 'FFN': 96, 'FullAttn': 32})
priority_counts: Counter({'P2-暂缓': 120, 'P0-优先': 96, 'P1-次优': 32})
saved_layer_list: /home/haozhong/ECE9483/layer_list.csv

representative benchmark rows:
- FFN_gate_proj          | FFN           | gate_proj    | [9216, 2560]   | model.language_model.layers.0.mlp.gate_proj
- FFN_down_proj          | FFN           | down_proj    | [2560, 9216]   | model.language_model.layers.0.mlp.down_proj
- FullAttn_q_proj        | FullAttn      | q_proj       | [8192, 2560]   | model.language_model.layers.3.self_attn.q_proj
- FullAttn_k_proj        | FullAttn      | k_proj       | [1024, 2560]   | model.language_model.layers.3.self_attn.k_proj
- DeltaNet_in_proj_qkv   | DeltaNet_Attn | in_proj_qkv  | [8192, 2560]   | model.language_model.layers.0.linear_attn.in_proj_qkv
- DeltaNet_in_proj_z     | DeltaNet_Attn | in_proj_z    | [4096, 2560]   | model.language_model.layers.0.linear_attn.in_proj_z


## Task 3: FP16 Baseline

In [5]:
print('注意: 本地 checkpoint 的真实层形状以 named_modules() 为准。')
print('例如 FullAttn q_proj 实际是 [8192, 2560]，DeltaNet 以 fused linear_attn 投影暴露。')

fp16_rows = []
for row in benchmark_rows:
    module = resolve_module(model, row['submodule_name'])
    weight = module.weight.detach()
    x = torch.randn(1, weight.shape[1], dtype=weight.dtype, device=weight.device)
    latency_ms = benchmark_single_layer_fp16(weight, x, n_warmup=10, n_runs=100)
    bytes_read = weight.shape[0] * weight.shape[1] * 2 + weight.shape[1] * 2
    bytes_write = weight.shape[0] * 2
    total_bytes = bytes_read + bytes_write
    bandwidth_gbs = total_bytes / (latency_ms / 1000.0) / 1e9
    result = {
        'benchmark_name': row['benchmark_name'],
        'sub_type': row['sub_type'],
        'proj_name': row['proj_name'],
        'submodule_name': row['submodule_name'],
        'M': 1,
        'N': int(weight.shape[0]),
        'K': int(weight.shape[1]),
        'latency_ms': latency_ms,
        'bandwidth_GBs': bandwidth_gbs,
        'total_bytes_MB': total_bytes / 1e6,
    }
    fp16_rows.append(result)
    print(
        f"{result['benchmark_name']:22s} [{result['N']:5d}, {result['K']:5d}] -> "
        f"{result['latency_ms']:.4f} ms, BW={result['bandwidth_GBs']:.1f} GB/s, "
        f"data={result['total_bytes_MB']:.2f} MB"
    )

baseline_fp16_path = Path('baseline_fp16.csv')
with baseline_fp16_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=['benchmark_name', 'sub_type', 'proj_name', 'submodule_name', 'M', 'N', 'K', 'latency_ms', 'bandwidth_GBs', 'total_bytes_MB'],
    )
    writer.writeheader()
    writer.writerows(fp16_rows)
print('saved_baseline_fp16:', baseline_fp16_path.resolve())

注意: 本地 checkpoint 的真实层形状以 named_modules() 为准。
例如 FullAttn q_proj 实际是 [8192, 2560]，DeltaNet 以 fused linear_attn 投影暴露。
FFN_gate_proj          [ 9216,  2560] -> 0.0958 ms, BW=493.0 GB/s, data=47.21 MB
FFN_down_proj          [ 2560,  9216] -> 0.0917 ms, BW=515.0 GB/s, data=47.21 MB
FullAttn_q_proj        [ 8192,  2560] -> 0.0863 ms, BW=486.5 GB/s, data=41.96 MB
FullAttn_k_proj        [ 1024,  2560] -> 0.0121 ms, BW=434.5 GB/s, data=5.25 MB
DeltaNet_in_proj_qkv   [ 8192,  2560] -> 0.0811 ms, BW=517.4 GB/s, data=41.96 MB
DeltaNet_in_proj_z     [ 4096,  2560] -> 0.0405 ms, BW=518.3 GB/s, data=20.98 MB
saved_baseline_fp16: /home/haozhong/ECE9483/baseline_fp16.csv


In [6]:
print('Running auxiliary end-to-end baseline ...')
end_to_end_rows = build_baseline_rows(model, tokenizer, gen_tokens=64)
pprint(end_to_end_rows)

combined_rows = []
for row in end_to_end_rows:
    combined_rows.append(
        {
            'record_type': 'end_to_end',
            'scenario': row['scenario'],
            'input_tokens': row['input_tokens'],
            'prefill_latency_s': row['prefill_latency_s'],
            'decode_latency_s_per_token': row['decode_latency_s_per_token'],
            'peak_vram_mib': row['peak_vram_mib'],
            'M': '',
            'N': '',
            'K': '',
            'fp16_latency_ms': '',
        }
    )
for row in fp16_rows:
    combined_rows.append(
        {
            'record_type': 'single_layer',
            'scenario': row['benchmark_name'],
            'input_tokens': '',
            'prefill_latency_s': '',
            'decode_latency_s_per_token': '',
            'peak_vram_mib': '',
            'M': row['M'],
            'N': row['N'],
            'K': row['K'],
            'fp16_latency_ms': row['latency_ms'],
        }
    )

baseline_results_path = Path('baseline_results.csv')
with baseline_results_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=['record_type', 'scenario', 'input_tokens', 'prefill_latency_s', 'decode_latency_s_per_token', 'peak_vram_mib', 'M', 'N', 'K', 'fp16_latency_ms'],
    )
    writer.writeheader()
    writer.writerows(combined_rows)
print('saved_baseline_results:', baseline_results_path.resolve())

Running auxiliary end-to-end baseline ...


[{'decode_latency_s_per_token': 0.042063655562500205,
  'input_tokens': 133,
  'peak_vram_mib': 8739.13037109375,
  'prefill_latency_s': 0.23657844489998753,
  'scenario': 'short_128'},
 {'decode_latency_s_per_token': 0.04234676384375149,
  'input_tokens': 527,
  'peak_vram_mib': 8826.07470703125,
  'prefill_latency_s': 0.2793902198999604,
  'scenario': 'mid_512'},
 {'decode_latency_s_per_token': 0.04485894952187479,
  'input_tokens': 2060,
  'peak_vram_mib': 9182.88134765625,
  'prefill_latency_s': 0.6022108355000227,
  'scenario': 'long_2048'}]
saved_baseline_results: /home/haozhong/ECE9483/baseline_results.csv


## Task 4: 量化脚本

In [7]:
quantized_path = Path('quantized_weights.pt')
quantized_results = {}
rebuild_quantized = True

if quantized_path.exists():
    print('Found cached quantized artifact, loading ...')
    quantized_results = attach_layer_metadata(load_quantized(quantized_path), rows)
    rebuild_quantized = len(quantized_results) != len(rows) or any(
        info.get('sub_type') is None for info in quantized_results.values()
    )
    print('cached_layers:', len(quantized_results))
    print('cache_covers_all_phase1_layers:', not rebuild_quantized)

if rebuild_quantized:
    print('Rebuilding quantized_weights.pt for all phase-one target layers ...')
    quantized_results = quantize_layer_records(model, rows)
else:
    print('Cache is reusable; only refreshing metadata and re-saving.')

save_quantized(quantized_results, quantized_path)
print('saved_quantized:', quantized_path.resolve())
print('quantized_layers:', len(quantized_results))

error_summary = summarize_errors_by_type(quantized_results)
print()
print('error_summary_by_sub_type:')
pprint(error_summary)

worst_layers = sorted(
    quantized_results.items(),
    key=lambda item: item[1]['errors']['max_abs_err'],
    reverse=True,
)[:5]
print()
print('worst_5_layers_by_max_abs_err:')
for name, info in worst_layers:
    print(
        f"- {name} | sub_type={info.get('sub_type')} | proj={info.get('proj_name')} | "
        f"shape={info['shape']} | max_abs_err={info['errors']['max_abs_err']:.6f} | "
        f"mean_rel_err={info['errors']['mean_rel_err']:.6f}"
    )

Found cached quantized artifact, loading ...


cached_layers: 96
cache_covers_all_phase1_layers: False
Rebuilding quantized_weights.pt for all phase-one target layers ...


saved_quantized: /home/haozhong/ECE9483/quantized_weights.pt
quantized_layers: 248

error_summary_by_sub_type:
[{'avg_max_abs_err': 0.0009795347849527994,
  'avg_mean_abs_err': 0.00014640661920566344,
  'avg_mean_rel_err': 0.0540117091499269,
  'count': 120,
  'sub_type': 'DeltaNet_Attn'},
 {'avg_max_abs_err': 0.0009776502847671509,
  'avg_mean_abs_err': 7.615729896315315e-05,
  'avg_mean_rel_err': 0.04198599459292988,
  'count': 96,
  'sub_type': 'FFN'},
 {'avg_max_abs_err': 0.001186072826385498,
  'avg_mean_abs_err': 0.0001191391152133292,
  'avg_mean_rel_err': 0.048443158622831106,
  'count': 32,
  'sub_type': 'FullAttn'}]

worst_5_layers_by_max_abs_err:
- model.language_model.layers.23.self_attn.o_proj | sub_type=FullAttn | proj=o_proj | shape=[2560, 4096] | max_abs_err=0.002720 | mean_rel_err=0.047296
- model.language_model.layers.27.self_attn.o_proj | sub_type=FullAttn | proj=o_proj | shape=[2560, 4096] | max_abs_err=0.002666 | mean_rel_err=0.046992
- model.language_model.layers.

## Task 5: CPU Reference

In [8]:
reference_targets = [
    row for row in benchmark_rows
    if row['benchmark_name'] in {'FFN_gate_proj', 'FullAttn_q_proj', 'DeltaNet_in_proj_qkv'}
]

actual_layer_reports = []
for row in reference_targets:
    module = resolve_module(model, row['submodule_name'])
    weight_cpu = module.weight.detach().float().cpu()
    x = torch.randn(1, weight_cpu.shape[1], dtype=torch.float16)
    qinfo = quantized_results[row['submodule_name']]
    y_ref = x.float() @ weight_cpu.T
    y_quant = quantized_matmul_reference(x, qinfo['qweight'], qinfo['scale'])
    actual_layer_reports.append(check_correctness(y_ref, y_quant, tag=row['benchmark_name']))

gemv_row = reference_targets[0]
gemv_module = resolve_module(model, gemv_row['submodule_name'])
gemv_weight_cpu = gemv_module.weight.detach().float().cpu()
gemv_x = torch.randn(gemv_weight_cpu.shape[1], dtype=torch.float16)
gemv_info = quantized_results[gemv_row['submodule_name']]
gemv_ref = gemv_weight_cpu @ gemv_x.float()
gemv_quant = quantized_matvec_reference(gemv_x, gemv_info['qweight'], gemv_info['scale'])
gemv_report = check_correctness(gemv_ref, gemv_quant, tag=f"{gemv_row['benchmark_name']}_gemv")

print()
print('actual_layer_reports:')
pprint(actual_layer_reports)
print()
print('actual_gemv_report:')
pprint(gemv_report)
print()
print('synthetic_cpu_reference_self_test:')
synthetic_reports = self_test()
pprint(synthetic_reports)

[FFN_gate_proj] max=0.049163 mean=0.003074 rel=0.076364 cos=0.99996138
[FullAttn_q_proj] max=0.063881 mean=0.005241 rel=0.076459 cos=0.99995601


[DeltaNet_in_proj_qkv] max=0.073102 mean=0.005079 rel=0.135801 cos=0.99994254
[FFN_gate_proj_gemv] max=0.029124 mean=0.003083 rel=0.043503 cos=0.99996209

actual_layer_reports:
[{'cos_sim': 0.9999613761901855,
  'max_abs_err': 0.04916346073150635,
  'mean_abs_err': 0.003074248554185033,
  'rel_err': 0.07636400312185287,
  'tag': 'FFN_gate_proj'},
 {'cos_sim': 0.9999560117721558,
  'max_abs_err': 0.06388086080551147,
  'mean_abs_err': 0.005240844562649727,
  'rel_err': 0.07645896077156067,
  'tag': 'FullAttn_q_proj'},
 {'cos_sim': 0.9999425411224365,
  'max_abs_err': 0.07310163974761963,
  'mean_abs_err': 0.005079310387372971,
  'rel_err': 0.1358010619878769,
  'tag': 'DeltaNet_in_proj_qkv'}]

actual_gemv_report:
{'cos_sim': 0.9999620914459229,
 'max_abs_err': 0.02912449836730957,
 'mean_abs_err': 0.0030829841271042824,
 'rel_err': 0.04350346699357033,
 'tag': 'FFN_gate_proj_gemv'}

synthetic_cpu_reference_self_test:


[FFN_gate_proj] max=1.946003 mean=0.324156 rel=0.063463 cos=0.99996728


[FullAttn_q_proj] max=1.588173 mean=0.328821 rel=0.060745 cos=0.99996716


[DeltaNet_in_proj_qkv] max=1.648132 mean=0.316222 rel=0.050769 cos=0.99996626
Self test PASSED
[{'cos_sim': 0.9999672770500183,
  'max_abs_err': 1.9460029602050781,
  'mean_abs_err': 0.32415610551834106,
  'rel_err': 0.0634625256061554,
  'tag': 'FFN_gate_proj'},
 {'cos_sim': 0.9999671578407288,
  'max_abs_err': 1.5881729125976562,
  'mean_abs_err': 0.328821063041687,
  'rel_err': 0.0607445202767849,
  'tag': 'FullAttn_q_proj'},
 {'cos_sim': 0.9999662637710571,
  'max_abs_err': 1.64813232421875,
  'mean_abs_err': 0.31622153520584106,
  'rel_err': 0.05076911672949791,
  'tag': 'DeltaNet_in_proj_qkv'}]
